# Getting Data

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## stdin and stdout

When you run a Python script from the command line (terminal / Command Prompt):
- stdin ("standard input") = the data coming into the script
- stdout ("standard output") = the data going out of the script

Using the pipe symbol `|`, the output of one command becomes the input of the next, like water flowing through connected pipes:

cat chai_shop_log.txt | python egrep.py "[0-9]" | python line_count.py

read the file -> keep only the lines with numbers -> count those lines.

In [3]:
from pathlib import Path

# All practice files for this chapter will go into this folder
log_text = """Monday: sold 120 cups of chai
Tea leaves delivered for the chai
Tuesday: sold 135 cups of chai
Milk was late in the morning
Wednesday: sold 110 cups of chai and 40 samosas
Shop closed early due to rain
Thursday: sold 150 cups chai
Friday: 160 chai sold
"""
data = Path("getting_data")
data.mkdir(exist_ok=True)
(data / "chai_shop_log.txt").write_text(log_text)
print(log_text)

Monday: sold 120 cups of chai
Tea leaves delivered for the chai
Tuesday: sold 135 cups of chai
Milk was late in the morning
Wednesday: sold 110 cups of chai and 40 samosas
Shop closed early due to rain
Thursday: sold 150 cups chai
Friday: 160 chai sold



In [5]:
%%writefile getting_data/egrep.py
# egrep.py: print only the lines that match a pattern (regex)
import sys, re

regex = sys.argv[1]            # sys.argv[1] = the pattern given on the command line

for line in sys.stdin:         # read every line coming IN
    if re.search(regex, line):
        sys.stdout.write(line) # send matching lines OUT

Overwriting getting_data/egrep.py


In [7]:
%%writefile getting_data/line_count.py
# line_count.py: count the lines coming in
import sys

count = 0
for line in sys.stdin:
    count += 1

print(count)  # print() also goes to stdout

Overwriting getting_data/line_count.py


In [8]:
%%writefile getting_data/most_common_words.py
# most_common_words.py: show the N most common words
import sys
from collections import Counter

try:
    num_words = int(sys.argv[1])
except (IndexError, ValueError):
    print("usage: most_common_words.py num_words")
    sys.exit(1)                # non-zero exit code = error

counter = Counter(word.lower()
                  for line in sys.stdin
                  for word in line.strip().split()
                  if word)

for word, count in counter.most_common(num_words):
    sys.stdout.write(f"{count}\t{word}\n")

Overwriting getting_data/most_common_words.py


## Reading Files

### Opening a file
- `'r'` : read only
- `'w'`: write and also deletes the old contents if the file exists! 
- `'a'`: append (add to the end)

Always use a `with` block. It closes the file automatically, even if there is an error:

```python
with open("file.txt", "r") as f:
       data = f.read()
```
here the file is already closed


In [9]:
from collections import Counter
emails_text = """priya.sharma@gmail.com
rahul@yahoo.co.in
anita@gmail.com
vikram@rediffmail.com
sneha@company.co.in
this line has no email address
arjun@GMAIL.com
meera@yahoo.co.in
ravi@mail.company.co.in
"""
(data / "email_addresses.txt").write_text(emails_text)

def get_domain(email_address):
    """Split on '@' and return the last piece (in lowercase)"""
    return email_address.lower().split("@")[-1]

with open(data / "email_addresses.txt", "r") as f:
    domain_counts = Counter(get_domain(line.strip())    # strip() removes the "\n" at the end
                            for line in f
                            if "@" in line)              # skip lines without an email

print(domain_counts.most_common())

[('gmail.com', 3), ('yahoo.co.in', 2), ('rediffmail.com', 1), ('company.co.in', 1), ('mail.company.co.in', 1)]


## Delimited Files (CSV / TSV)

Most data files have many values on each line, separated by a delimiter:
- **CSV** = Comma-Separated Values (`,`)
- **TSV** = Tab-Separated Values (`\t`)

Never split these lines yourself with `.split(",")`. Values can **contain** commas, tabs or even new lines! Use Python's `csv` module or pandas, which handle all of this properly.

### Example 1: a file with no header (tab-separated)
Daily vegetable prices at a local mandi (made-up numbers, ₹ per kg):

In [10]:
import csv
(data / "mandi_prices.txt").write_text(
    "2026-10-19\tTomato\t40\n"
    "2026-10-19\tOnion\t35\n"
    "2026-10-19\tPotato\t25\n"
    "2026-10-20\tTomato\t45\n"
    "2026-10-20\tOnion\t38\n"
    "2026-10-20\tPotato\t24\n"
)

# csv.reader gives each row as a LIST
with open(data / "mandi_prices.txt", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        date, vegetable, price = row[0], row[1], float(row[2])
        print(f"{date}  {vegetable:<7} ₹{price:.0f}/kg")

2026-10-19  Tomato  ₹40/kg
2026-10-19  Onion   ₹35/kg
2026-10-19  Potato  ₹25/kg
2026-10-20  Tomato  ₹45/kg
2026-10-20  Onion   ₹38/kg
2026-10-20  Potato  ₹24/kg


In [12]:
# With a header row, csv.DictReader gives each row as a dictionary 
# so you can use column names instead of numbers.

(data / "mandi_prices_colon.txt").write_text(
    "date:vegetable:price_per_kg\n"
    "2026-10-19:Tomato:40\n"
    "2026-10-19:Onion:35\n"
    "2026-10-19:Potato:25\n"
)

with open(data / "mandi_prices_colon.txt", newline="") as f:
    reader = csv.DictReader(f, delimiter=":")
    for row in reader:
        print(row["vegetable"], "-> ₹", float(row["price_per_kg"]))

Tomato -> ₹ 40.0
Onion -> ₹ 35.0
Potato -> ₹ 25.0


In [15]:
# No header -> give the column names ourselves
prices = pd.read_csv(data / "mandi_prices.txt", sep="\t", header=None,
                     names=["date", "vegetable", "price"])
print(prices, "\n")

# With a header → pandas reads the names automatically
prices_colon = pd.read_csv(data / "mandi_prices_colon.txt", sep=":")
print(prices_colon, "\n")

# And analysis is easy: average price of each vegetable
print(prices.groupby("vegetable")["price"].mean())

         date vegetable  price
0  2026-10-19    Tomato     40
1  2026-10-19     Onion     35
2  2026-10-19    Potato     25
3  2026-10-20    Tomato     45
4  2026-10-20     Onion     38
5  2026-10-20    Potato     24 

         date vegetable  price_per_kg
0  2026-10-19    Tomato            40
1  2026-10-19     Onion            35
2  2026-10-19    Potato            25 

vegetable
Onion     36.5
Potato    24.5
Tomato    42.5
Name: price, dtype: float64


In [16]:
items = [["Tomato", "red, ripe", 40],
         ["Onion", "big, red", 35],
         ["Potato", "new crop", 25]]

# DON'T do this: joining with commas by hand
with open(data / "bad.csv", "w") as f:
    for row in items:
        f.write(",".join(map(str, row)) + "\n")

# DO this: csv.writer adds quotes where needed
with open(data / "good.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["vegetable", "description", "price"])
    writer.writerows(items)

print("bad.csv:\n" + (data / "bad.csv").read_text())
print("good.csv:\n" + (data / "good.csv").read_text())

# Reading them back
with open(data / "bad.csv", newline="") as f:
    print("bad.csv first row has", len(next(csv.reader(f))), "columns X (should be 3)")
print("good.csv read by pandas:")
print(pd.read_csv(data / "good.csv"))

bad.csv:
Tomato,red, ripe,40
Onion,big, red,35
Potato,new crop,25

good.csv:
vegetable,description,price
Tomato,"red, ripe",40
Onion,"big, red",35
Potato,new crop,25

bad.csv first row has 4 columns X (should be 3)
good.csv read by pandas:
  vegetable description  price
0    Tomato   red, ripe     40
1     Onion    big, red     35
2    Potato    new crop     25


In [17]:
# pandas writes a correct CSV in one line too
pd.DataFrame(items, columns=["vegetable", "description", "price"]).to_csv(data / "good_pandas.csv", index=False)
print((data / "good_pandas.csv").read_text())

vegetable,description,price
Tomato,"red, ripe",40
Onion,"big, red",35
Potato,new crop,25



## Scraping the Web

**Web scraping** = getting data out of **web pages**. Downloading a page is easy; pulling **useful, structured** information out of it is the hard part.

### HTML
Web pages are written in **HTML**. Text is wrapped in **tags** like `<p>…</p>`, and tags can have **attributes** like `id` or `class`:

```html
<p id="subject" class="important">Data Science</p>
```

We use two libraries:
| Library | Job | Install |
|---|---|---|
| **requests** | Download the web page | `pip install requests` |
| **BeautifulSoup** | Read (parse) the HTML and find things in it | `pip install beautifulsoup4` |

In [4]:
from bs4 import BeautifulSoup

html = """
<html>
  <head><title>My Data Page</title></head>
  <body>
    <p id="author">Debasis Das</p>
    <p id="subject" class="important">Data Science</p>
    <p class="important">Python is fun</p>
    <p>Just a normal line</p>
    <div><span>Bhubaneswar</span><span>Noida</span></div>
    <span>Not inside a div</span>
  </body>
</html>
"""
soup = BeautifulSoup(html, "html.parser")

print("Page title:          ", soup.title.text)
print("First <p>:           ", soup.find("p"))                   # or just soup.p
print("Its text:            ", soup.p.text)
print("Its id:              ", soup.p["id"])                      # KeyError if there is no id
print("Safe get (no class): ", soup.p.get("class"))               # None if missing
print("Number of <p> tags:  ", len(soup.find_all("p")))           # or soup("p")
print("<p> with an id:      ", [p.text for p in soup("p") if p.get("id")])
print("class='important':   ", [p.text for p in soup("p", "important")])
print("<span> inside <div>: ", [span.text for div in soup("div") for span in div("span")])

Page title:           My Data Page
First <p>:            <p id="author">Debasis Das</p>
Its text:             Debasis Das
Its id:               author
Safe get (no class):  None
Number of <p> tags:   4
<p> with an id:       ['Debasis Das', 'Data Science']
class='important':    ['Data Science', 'Python is fun']
<span> inside <div>:  ['Bhubaneswar', 'Noida']
